## **P2: Importing Dataset & Data Wrangling**

Using GEO ((Gene Expression Omnibus) dataset GSE13164 for leukemia diagnosis, includes:
- Downloading and parsing GEO series data
- Extracting and filtering labels
- Merging sample expression data
- Mapping probes to gene accessions
- Aligning and encoding the final dataset

### **1. Import Required Libraries**

Load all necessary libraries for data processing:
- **GEOparse**: To access to sample metadata, expression data, and platform information.
- **pandas (pd)**: To create DataFrames, merge tables, and handle CSV input/output.
- **numpy (np)**: For array operations and mathematical functions.
- **LabelEncoder (from sklearn.preprocessing)**: To convert categorical labels (leukemia types) into numeric codes for machine learning models.
- **re**: To identify leukemia types in sample characteristics using regex search.
- **os**: To create directories and manage file paths.
- **reduce (from functools)**: To merge multiple DataFrames sequentially.

In [15]:
import GEOparse
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import re 
import os 
from functools import reduce

### **2. Configuration and Constants**

Set up configuration parameters including the GEO dataset ID, output file names, target disease classes, and feature identifier column.

In [16]:
# configuration
GSE_ID = "GSE13164" 
OUTPUT_CLEAN_DATA = "GSE13164_cleaned_features.csv"
OUTPUT_CLEAN_LABELS = "GSE13164_cleaned_labels.csv"

# constants
TARGET_CLASSES = ['ALL', 'AML', 'CLL', 'CML']
FEATURE_IDENTIFIER = 'GB_ACC'

### **3. Download and Parse GEO Data**

Downloads the GEO dataset using GEOparse and extracts the platform (GPL) annotation table. Returns both the GEO object (containing sample data) and the platform table (containing probe annotations).

In [17]:
def download_and_parse_geo(gse_id):
    print(f"Downloading and parsing GEO series: {gse_id}...")
    print(f"Current working directory: {os.getcwd()}")
    
    try:
        # Try to load from local Raw Data folder first if it exists
        local_file = f"./Raw Data/{gse_id}_family.soft.gz"
        print(f"Checking for local file at: {local_file}")
        print(f"Local file exists: {os.path.exists(local_file)}")
        
        if os.path.exists(local_file):
            print(f"Loading GEO data from local file: {local_file}")
            gse = GEOparse.get_GEO(filepath=local_file, silent=False)
        else:
            print(f"Local file not found, attempting FTP download...")
            gse = GEOparse.get_GEO(geo=gse_id, destdir="./", silent=False)
    except Exception as e:
        print(f"Error downloading/parsing {gse_id}: {e}")
        return None, None
    
    platform_key = list(gse.gpls.keys())[0]
    gpl_table = gse.gpls[platform_key].table.copy()
    
    return gse, gpl_table

### **4. Extract and Filter Samples**

Iterates through all samples in the GEO dataset and:
- Extracts leukemia type labels from sample characteristics
- Filters for only target classes (ALL, AML, CLL, CML)
- Extracts expression values for valid samples
- Returns filtered sample DataFrames and metadata

In [18]:
def extract_and_filter_samples(gse):
    metadata_list = []
    sample_dfs = []

    for name, gsm in gse.gsms.items():
        characteristics = ' '.join(gsm.metadata.get('characteristics_ch1', ['']))
        leukemia_type = None
        
        # identify leukemia type based on target classes
        for t_type in TARGET_CLASSES:
            if re.search(r'\b' + re.escape(t_type) + r'\b', characteristics, re.IGNORECASE):
                leukemia_type = t_type
                break

        # only process samples belonging to target classes
        if leukemia_type in TARGET_CLASSES:
            metadata_list.append({'Sample_ID': name, 'Leukemia_Type': leukemia_type})
            
            # extract expression data
            gsm_df = gsm.table.copy()
            
            if 'ID_REF' not in gsm_df.columns:
                print(f"Warning: GSM {name} missing 'ID_REF'. Skipping.")
                continue
            
            # identify expression val col
            value_col = next((col for col in gsm_df.columns if col.upper() in ['VALUE', 'LOG_RATIO', 'SIGNAL', 'AVG_SIGNAL', 'NORMALIZED_SIGNAL']), None)
            
            if not value_col:
                non_id_cols = [c for c in gsm_df.columns if c != 'ID_REF']
                if non_id_cols:
                    value_col = non_id_cols[-1]
                else:
                    print(f"Warning: GSM {name} has no detectable value column. Skipping.")
                    continue

            gsm_df = gsm_df[['ID_REF', value_col]].rename(columns={value_col: name})
            sample_dfs.append(gsm_df)

    metadata_df = pd.DataFrame(metadata_list)
    print(f"Samples identified and labeled: {len(sample_dfs)}.")
    
    return sample_dfs, metadata_df

### **5. Merge Expression Data**

Combines all individual sample DataFrames into a single expression matrix by merging on probe ID (ID_REF). Uses an inner join to retain only probes that are present in all samples.

In [19]:
def merge_expression_data(sample_dfs):
    expression_data = reduce(lambda left, right: pd.merge(left, right, on='ID_REF', how='inner'), sample_dfs)
    expression_data = expression_data.set_index('ID_REF')
    
    print(f"Initial Merged Feature Matrix Shape (Probes x Samples): {expression_data.shape}")
    
    return expression_data

### **6. Annotate and Aggregate Features**

Maps probes to gene accessions using the platform annotation table and aggregates expression values by gene:
- Extracts and cleans feature IDs (gene accessions) from platform data
- Merges annotation with expression data
- Groups by gene and computes mean expression across probes
- Transposes to create a sample × gene feature matrix

In [20]:
def annotate_and_aggregate_features(expression_data, gpl_table):
    # extract annotation information from platform table
    annotation_cols = ['ID', FEATURE_IDENTIFIER]
    annotation_df = gpl_table[annotation_cols].rename(columns={'ID': 'ID_REF', FEATURE_IDENTIFIER: 'Feature_ID'})

    # clean feature IDs
    annotation_df.dropna(subset=['Feature_ID'], inplace=True)
    annotation_df = annotation_df[annotation_df['Feature_ID'].str.strip() != '---']
    annotation_df['Feature_ID'] = annotation_df['Feature_ID'].apply(lambda x: x.split(' // ')[0].strip())
    
    # merge annotation with expression data
    merged_df = pd.merge(expression_data.reset_index(), annotation_df, on='ID_REF', how='inner')
    merged_df.dropna(subset=['Feature_ID'], inplace=True)
    
    # get sample cols & aggregate by gene
    sample_cols = [col for col in merged_df.columns if col.startswith('GSM')]
    final_features_df = merged_df.groupby('Feature_ID')[sample_cols].mean()
    
    # transpose, samples as rows, features as cols
    final_features_df = final_features_df.T
    
    return final_features_df

### **7. Align and Encode Labels**

Prepares the final dataset by:
- Aligning metadata with the feature matrix using sample IDs
- Encoding categorical leukemia types to numeric codes (0, 1, 2, 3)
- Verifying sample-wise alignment between features and labels

In [21]:
def align_and_encode_labels(final_features_df, metadata_df):
    # align metadata with sample id
    metadata_df = metadata_df.set_index('Sample_ID').loc[final_features_df.index.tolist()].reset_index()

    # encode target labels
    le = LabelEncoder()
    metadata_df['Target_Code'] = le.fit_transform(metadata_df['Leukemia_Type'])

    # final alignment check
    print("\n--- Final Data Alignment ---")
    print(f"Final Feature Matrix Shape (Samples x Features): {final_features_df.shape}")
    print(f"Final Label Matrix Shape (Samples x Info): {metadata_df.shape}")
    print(f"Sample-wise alignment check (should be TRUE): {all(final_features_df.index == metadata_df['Sample_ID'])}")

    return final_features_df, metadata_df


### **8. Functions Calling**

Coordinates the entire data processing pipeline (from 3-7) by calling each helper function in sequence.


In [22]:
def load_and_preprocess_geo(gse_id):
    # 3. download & parse
    gse, gpl_table = download_and_parse_geo(gse_id)
    if gse is None:
        return None, None
    
    #4. extract & filter samples
    sample_dfs, metadata_df = extract_and_filter_samples(gse)
    if sample_dfs is None:
        return None, None
    
    # 5. merge expression data
    expression_data = merge_expression_data(sample_dfs)
    
    # 6. annotate & aggregate
    final_features_df = annotate_and_aggregate_features(expression_data, gpl_table)
    
    # 7. align & encode
    final_features_df, metadata_df = align_and_encode_labels(final_features_df, metadata_df)
    
    return final_features_df, metadata_df

### **9. Data Processing and Export**

Execute the main pipeline:
- Create output directory for processed data
- Run the complete GEO data processing function
- Save cleaned features and labels to CSV files

In [23]:
# create data directory
os.makedirs('data', exist_ok=True)

print("--- Starting O1: Data Collection & Wrangling (Python) ---")
# load, clean & map the data
expression_data, metadata = load_and_preprocess_geo(GSE_ID)

if expression_data is not None and metadata is not None:
    try:
        # save the final products for next step
        expression_data.to_csv(OUTPUT_CLEAN_DATA)
        metadata.to_csv(OUTPUT_CLEAN_LABELS, index=False)
        print(f"\nSUCCESS: Cleaned features saved to: {OUTPUT_CLEAN_DATA}")
        print(f"SUCCESS: Cleaned labels saved to: {OUTPUT_CLEAN_LABELS}")
    except Exception as e:
        print(f"Error saving data: {e}")

    print("Data Wrangling (O1) is now complete.")

13-Jan-2026 18:10:37 INFO GEOparse - Parsing ./Raw Data/GSE13164_family.soft.gz: 
13-Jan-2026 18:10:37 DEBUG GEOparse - DATABASE: GeoMiame
13-Jan-2026 18:10:37 DEBUG GEOparse - SERIES: GSE13164
13-Jan-2026 18:10:37 DEBUG GEOparse - PLATFORM: GPL7473
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331733
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331734
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331735
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331736
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331737
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331738
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331739
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331740
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331741
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331742
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331743
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331744
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331745
13-Jan-2026 18:10:37 D

--- Starting O1: Data Collection & Wrangling (Python) ---
Current working directory: /Users/nguyuling/Leukemia-Diagnosis
Checking for local file at: ./Raw Data/GSE13164_family.soft.gz
Local file exists: True
Loading GEO data from local file: ./Raw Data/GSE13164_family.soft.gz


13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331810
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331811
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331812
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331813
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331814
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331815
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331816
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331817
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331818
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331819
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331820
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331821
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331822
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331823
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331824
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331825
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GSM331826
13-Jan-2026 18:10:37 DEBUG GEOparse - SAMPLE: GS

Samples identified and labeled: 973.
Initial Merged Feature Matrix Shape (Probes x Samples): (1480, 973)

--- Final Data Alignment ---
Final Feature Matrix Shape (Samples x Features): (973, 1420)
Final Label Matrix Shape (Samples x Info): (973, 3)
Sample-wise alignment check (should be TRUE): True

SUCCESS: Cleaned features saved to: GSE13164_cleaned_features.csv
SUCCESS: Cleaned labels saved to: GSE13164_cleaned_labels.csv
Data Wrangling (O1) is now complete.
